In [1]:
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()
MODEL = "gpt-4.1-nano"

kb_path = Path("..")/ "practice-kb" / "brightdesk_kb.md"
kb_text = kb_path.read_text(encoding="utf-8")

print("Characters: ", len(kb_text))
print(kb_text[:300])

Characters:  3822
# BrightDesk IT Solutions — Knowledge Base

## Company Overview
BrightDesk IT Solutions is a managed IT services company founded in 2018 and headquartered in Manchester, UK. The company supports around 120 small and medium-sized businesses across the North West of England. BrightDesk has 45 staff an


In [2]:
sections = kb_text.split("\n## ")

print("Number of pieces:", len(sections))
print("---")
print(sections[0])
print("---")
print(sections[1][:150])

Number of pieces: 15
---
# BrightDesk IT Solutions — Knowledge Base

---
Company Overview
BrightDesk IT Solutions is a managed IT services company founded in 2018 and headquartered in Manchester, UK. The company supports ar


In [3]:
knowledge = {}

for piece in sections[1:]:
    lines = piece.split("\n")
    title = lines[0].lower()
    knowledge[title] = piece

print("Sections stored:", len(knowledge))
for title in knowledge:
    print(title)

Sections stored: 14
company overview
employee: priya patel
employee: rahul patel
employee: emma clarke
employee: daniel hughes
employee: sophie wright
service: deskcare
service: cloudguard
service: securestart
policy: passwords
policy: laptops and devices
policy: remote access (vpn)
support hours and response times
frequently asked questions


In [4]:
def get_relevant_context(question):
    question = question.lower()
    context = []

    for title in knowledge:
        clean_title = title.replace(":", " ").replace("(", " ").replace(")", " ")
        words = clean_title.split()

        for word in words:
            if len(word) > 3 and word in question:
                context.append(knowledge[title])
                break

    return context

In [5]:
questions = [
    "What does Priya do?",
    "How much does CloudGuard cost?",
    "Who is Patel?",
    "How often is my data backed up?",
]

for q in questions:
    docs = get_relevant_context(q)
    print(q, "->", len(docs), "sections")

What does Priya do? -> 1 sections
How much does CloudGuard cost? -> 1 sections
Who is Patel? -> 2 sections
How often is my data backed up? -> 0 sections


In [6]:
SYSTEM_PROMPT = """You are an assistant for BrightDesk IT Solutions.
Answer questions using ONLY the context provided.
If the context contains several people or services that match the question,
briefly describe each one.
If the answer is not in the context at all, say you don't know."""


def answer_question(question):
    documents = get_relevant_context(question)

    context = ""
    for doc in documents:
        context = context + doc + "\n\n"

    user_message = "Context:\n" + context + "\nQuestion: " + question

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
    )
    return response.choices[0].message.content

In [7]:
questions = [
    "Who is Patel?",
    "How often is my data backed up?",
    "How do I connect to the VPN from home?",
]

for q in questions:
    print("Q:", q)
    print("A:", answer_question(q))
    print("---")

Q: Who is Patel?
A: Patel could refer to either Priya Patel or Rahul Patel. Priya Patel is the Head of Service Desk at the Manchester office, and Rahul Patel is a Cloud Engineer at the Leeds office.
---
Q: How often is my data backed up?
A: I don't have information about how often your data is backed up.
---
Q: How do I connect to the VPN from home?
A: I don't have information about connecting to the VPN from home in the provided context.
---


In [8]:
STOP_WORDS = ["what", "how", "does", "the", "from", "is", "my", "do", "i", "to", "who", "often"]

def get_relevant_context(question):
    question_words = question.lower().replace("?", "").split()
    context = []

    for title in knowledge:
        section_text = knowledge[title].lower()

        for word in question_words:
            if word not in STOP_WORDS and word in section_text:
                context.append(knowledge[title])
                break

    return context

In [10]:
questions = [
    "How often is my data backed up?",
    "How do I connect to the VPN from home?",
    "Who is Patel?",
]

for q in questions:
    docs = get_relevant_context(q)
    print(q, "->", len(docs), "sections")

How often is my data backed up? -> 8 sections
How do I connect to the VPN from home? -> 2 sections
Who is Patel? -> 2 sections


In [11]:
for q in questions:
    print("Q:", q)
    print("A:", answer_question(q))
    print("---")

Q: How often is my data backed up?
A: Your data is backed up every 4 hours with the CloudGuard service.
---
Q: How do I connect to the VPN from home?
A: To connect to the VPN from home, you need to use the company's VPN service, which requires multi-factor authentication (MFA) at every login. If you experience any VPN issues, they should be raised as a priority ticket.
---
Q: Who is Patel?
A: There are two Patels mentioned:

- Priya Patel, who is the Head of Service Desk at the Manchester office.
- Rahul Patel, who is a Cloud Engineer at the Leeds office.
---
